## objectives
- Create a pytorch dataset from the squared image folder
- Load imagent1k weights for resnet50 as a baseline

In [10]:
from collections import Counter

import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


In [11]:
data_dir = '/home/takayuki/Desktop/summer2025/plants/data/AquaticPlantLabData/squared'

train_percent = 0.8
batch_size = 1

random_seed = 8

In [12]:
torch.manual_seed(random_seed)

basic_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    
])

dataset = datasets.ImageFolder(data_dir, transform = basic_tf)

train_size = int(train_percent * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size], 
                                                              generator=torch.Generator().manual_seed(random_seed))
train_dataloader = DataLoader(train_dataset, batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size, shuffle=False)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}\n")

# print how many images and of which classes are in the train and test datasets

def count_classes(subset):
    # Get the labels for the indices in the subset
    labels = [subset.dataset.targets[i] for i in subset.indices]
    label_counts = Counter(labels)
    return label_counts

train_class_counts = count_classes(train_dataset)
test_class_counts = count_classes(test_dataset)
class_names = dataset.classes
max_len = max(len(name) for name in class_names)
for i, class_name in enumerate(class_names):
    train_count = train_class_counts.get(i, 0)
    test_count = test_class_counts.get(i, 0)
    class_name = class_name.ljust(max_len+1)
    print(f"{class_name}: Train: {train_count}\t, Test: {test_count}")


Train dataset size: 172
Test dataset size: 43

Brasenia schreberi       : Train: 6	, Test: 3
Cabomba                  : Train: 2	, Test: 3
Ceratophyllum demersum   : Train: 21	, Test: 0
Elodea canadensis        : Train: 11	, Test: 4
Heteranthera dubia       : Train: 6	, Test: 1
Hydrocharis morsus-ranae : Train: 8	, Test: 4
Myriophyllum sibiricum   : Train: 4	, Test: 0
Myriophyllum spicatum    : Train: 12	, Test: 1
Najas flexilis           : Train: 4	, Test: 1
Nitellopsis obtusa       : Train: 3	, Test: 1
Nuphar variegata         : Train: 3	, Test: 0
Nymphaea odorata         : Train: 2	, Test: 0
Potamogeton crispus      : Train: 15	, Test: 6
Potamogeton gramineus    : Train: 13	, Test: 4
Potamogeton illinoensis  : Train: 29	, Test: 6
Potamogeton natans       : Train: 8	, Test: 1
Potamogeton praelongus   : Train: 3	, Test: 0
Potamogeton richardsonii : Train: 4	, Test: 2
Potamogeton robbinsii    : Train: 7	, Test: 2
Ranunculus aquatilis     : Train: 3	, Test: 2
Vallisneria americana    : 

## Load pretrained model
- ResNet50 with ImageNet1K weights

In [13]:
from tqdm import tqdm
from datetime import datetime

import timm
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary


In [14]:
model = timm.create_model('resnet50', pretrained=True)
summary(model, input_size=(1, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 1000]                 --
├─Conv2d: 1-1                            [1, 64, 112, 112]         9,408
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         128
├─ReLU: 1-3                              [1, 64, 112, 112]         --
├─MaxPool2d: 1-4                         [1, 64, 56, 56]           --
├─Sequential: 1-5                        [1, 256, 56, 56]          --
│    └─Bottleneck: 2-1                   [1, 256, 56, 56]          --
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           4,096
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-3                    [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-4                  [1, 64, 56, 56]           36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 56, 56]           128
│    │    └─Identity: 3-6                [1, 64, 56, 56]           --
│ 

In [15]:
model.fc = nn.Linear(model.fc.in_features, len(dataset.classes))

for param in model.parameters():
    param.requires_grad = False
    
for param in model.fc.parameters():
    param.requires_grad = True
    
print({"Number of trainable parameters: ", sum(p.numel() for p in model.parameters() if p.requires_grad)})

{'Number of trainable parameters: ', 43029}


## Training Loop
- Tensorboard logging
- Save model checkpoints
- Save best models

In [20]:
# Parameters
run_name = "Larger Learning rate (rerun of lost run)"
learning_rate = 0.001
num_epochs = 10

checkpoint_interval = 5
validation_interval = 1

# Paths
checkpoint_path = "/home/takayuki/Desktop/summer2025/plants/training/checkpoints"
models_path = "/home/takayuki/Desktop/summer2025/plants/training/models"
base_logs_path = "/home/takayuki/Desktop/summer2025/plants/training/runs/resnet50"

os.makedirs(checkpoint_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)
os.makedirs(base_logs_path, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
print(f"Using device: {device}, {torch.cuda.get_device_name(device) if device.type == 'cuda' else 'CPU'}")

Using device: cuda, NVIDIA RTX A500 Laptop GPU


In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=learning_rate)

now = datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H-%M__%S")

model_name = model.name if hasattr(model, 'name') else model.__class__.__name__

hparams_dict = {
    "a_run_name": run_name,
    "learning_rate": learning_rate,
    "num_epochs": num_epochs,
    "batch_size": batch_size,
    "train_dataset_size": len(train_dataset),
    "test_dataset_size": len(test_dataset),
    "model_architecture": model_name,
    "optimizer": optimizer.__class__.__name__,
    "criterion": criterion.__class__.__name__,
    "timestamp": timestamp,
}

writer = SummaryWriter(log_dir=os.path.join(base_logs_path, f"{run_name}_{model_name}_{timestamp}"))

for key, value in hparams_dict.items():
    writer.add_text(key, str(value))

# Training loop
best_val_accuracy = 0.0
best_val_epoch = 0

print("Starting training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{num_epochs}", unit="batch"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / total
    train_accuracy = correct / total
    
    print(f"Training Loss: {train_loss:.4f}, Training Accuracy: {train_accuracy*100:.4f}%")
    
    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_accuracy, epoch)
    
    # Validation phase
    if (epoch + 1) % validation_interval == 0:
        model.eval()  
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():  
            
            for val_images, val_labels in test_dataloader:
                val_images, val_labels = val_images.to(device), val_labels.to(device)
                
                val_outputs = model(val_images)
                val_loss = criterion(val_outputs, val_labels)
                
                val_running_loss += val_loss.item() * val_images.size(0)
                _, val_predicted = torch.max(val_outputs.data, 1)
                val_total += val_labels.size(0)
                val_correct += (val_predicted == val_labels).sum().item()
        
        val_loss = val_running_loss / val_total
        val_accuracy = val_correct / val_total

        print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy*100:.4f}%")
        
        writer.add_scalar('Loss/validation', val_loss, epoch)
        writer.add_scalar('Accuracy/validation', val_accuracy, epoch)
        
        # best model
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_val_epoch = epoch + 1
            
            # Delete previous best model if exists
            best_model_name = f"best_model_{model_name}_{timestamp}.pth"
            best_model_file_path = os.path.join(models_path, best_model_name)
            
            if os.path.exists(best_model_file_path):
                os.remove(best_model_file_path)
            
            # Save the new one
            torch.save(model.state_dict(), best_model_file_path)
            print(f"Saved new best model: {best_model_file_path}")
        
        
        
        
    # Checkpoint saving phase
    if (epoch + 1) % checkpoint_interval == 0:
        checkpoint_name = f"checkpoint_epoch_{epoch + 1}_{model_name}_{timestamp}.pth"
        checkpoint_file_path = os.path.join(checkpoint_path, checkpoint_name)
        
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss, 
        }, checkpoint_file_path)
        
        print(f"Saved checkpoint: {checkpoint_file_path}")

print("\nTraining complete.")
  

Starting training...


Epoch 1/10:   0%|          | 0/172 [00:00<?, ?batch/s]

Epoch 1/10: 100%|██████████| 172/172 [00:36<00:00,  4.67batch/s]


Training Loss: 2.9845, Training Accuracy: 14.5349%
Saved new best model: /home/takayuki/Desktop/summer2025/plants/training/models/best_model_ResNet_2025-06-17_15-09__06.pth
Validation Loss: 32.2166, Validation Accuracy: 13.9535%


Epoch 2/10: 100%|██████████| 172/172 [00:38<00:00,  4.44batch/s]


...

Training Loss: 1.7122, Training Accuracy: 66.8605%
Validation Loss: 28.3713, Validation Accuracy: 13.9535%


Epoch 9/10: 100%|██████████| 172/172 [00:42<00:00,  4.04batch/s]


Training Loss: 1.5712, Training Accuracy: 77.3256%
Validation Loss: 18.7163, Validation Accuracy: 6.9767%


Epoch 10/10: 100%|██████████| 172/172 [00:41<00:00,  4.18batch/s]


Training Loss: 1.4404, Training Accuracy: 81.3953%
Validation Loss: 33.4394, Validation Accuracy: 13.9535%
Saved checkpoint: /home/takayuki/Desktop/summer2025/plants/training/checkpoints/checkpoint_epoch_10_ResNet_2025-06-17_15-09__06.pth

Training complete.


In [22]:
observations = f"Validation loss is actually decreasing with the larger learning rate. Max train acc = {train_accuracy*100:.4f}%, Max val acc = {val_accuracy*100:.4f}%"
hparams_dict["observations"] = observations
writer.add_text('Observations', observations)

# Save the final model
final_model_name = f"final_model_{model_name}_{timestamp}.pth"
final_model_file_path = os.path.join(models_path, final_model_name)
torch.save(model.state_dict(), final_model_file_path)
print(f"Saved final model: {final_model_file_path}")

# Write Metrics and Hyperparameters to TensorBoard
metrics = {
    "epochs_trained": epoch + 1,
    "final_train_loss": train_loss,
    "final_train_accuracy": train_accuracy,
    "final_val_loss": val_loss,
    "final_val_accuracy": val_accuracy,
    "best_val_accuracy": best_val_accuracy,
    "best_val_epoch": best_val_epoch,
}
writer.add_hparams(hparams_dict, metrics)
writer.flush()
writer.close()  

Saved final model: /home/takayuki/Desktop/summer2025/plants/training/models/final_model_ResNet_2025-06-17_15-09__06.pth
